# Validate and freeze the pair book

This notebook uses the candidates exported by notebook 04. It uses **days 501–700 only** to decide which discovery candidates improve the portfolio. It does not tune model parameters; that happens in notebook 06. Days 701–1000 are never loaded.

## Validation workflow

1. Start with the discovery survivors from days 1–500.
2. Test each candidate alone and as an incremental addition on days 501–700.
3. Evaluate every eligible addition; retain it only if it improves validation score and does not reuse a selected ticker.
4. Export the frozen book for notebook 06.

The addition screen holds the trading configuration fixed. That isolates the question: **does this pair add value?**

In [52]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

DISCOVERY_END = 500
VALIDATION_END = 700
DOLLAR_LIMIT = 10_000

# Original-workflow prototype used only while comparing pair additions.
ADDITION_LOOKBACK = 150
ADDITION_ENTRY_Z = 1.00
ADDITION_EXIT_Z = 0.50
ADDITION_MAX_HOLDING_DAYS = 10_000  # effectively no time stop during pair selection

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'backtester').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from inside the project directory.')
sys.path.insert(0, str(repo_root / 'backtester'))

from backtest.engine import run_backtest
from backtest.metrics import compute_metrics

selection_path = repo_root / 'research_outputs' / 'pair_discovery_candidates.csv'
if not selection_path.exists():
    raise FileNotFoundError('Run every cell in notebooks/04_cointegration_pairs.ipynb first.')

pair_candidates = pd.read_csv(selection_path)
assert pair_candidates['discovery_end_day'].eq(DISCOVERY_END).all()
price_path = repo_root / 'backtester' / 'data' / '2026' / 'prices.txt'
price_frame = pd.read_csv(price_path, sep=r'\s+', nrows=VALIDATION_END)
prices = price_frame.to_numpy(dtype=float).T
ticker_to_index = {ticker: i for i, ticker in enumerate(price_frame.columns)}

core_pairs = [tuple(row) for row in pair_candidates.loc[pair_candidates['candidate_stage'].eq('core'), ['ticker_a', 'ticker_b']].itertuples(index=False, name=None)]
candidate_queue = pair_candidates.loc[pair_candidates['candidate_stage'].eq('validate_addition')].copy()

assert len(core_pairs) > 0
assert len({ticker for pair in core_pairs for ticker in pair}) == 2 * len(core_pairs)
print(f'Loaded {len(core_pairs)} core pairs and {len(candidate_queue)} validation candidates.')

Loaded 2 core pairs and 18 validation candidates.


## 1. A simple fixed-beta validation harness

Each candidate is evaluated with the same simple log-spread strategy. Its beta, intercept, spread mean, and spread volatility are calibrated at the first validation call using history through day 500, then held fixed for that run.

In [53]:
def make_fixed_beta_strategy(pair_book, lookback_days, entry_z, exit_z, max_holding_days):
    indexed_pairs = [(ticker_to_index[a], ticker_to_index[b], a, b) for a, b in pair_book]
    state = np.zeros(len(indexed_pairs), dtype=int)
    held_days = np.zeros(len(indexed_pairs), dtype=int)
    parameters = None
    last_history_length = None

    def reset_state():
        nonlocal state, held_days, parameters, last_history_length
        state = np.zeros(len(indexed_pairs), dtype=int)
        held_days = np.zeros(len(indexed_pairs), dtype=int)
        parameters = None
        last_history_length = None

    def get_position(prc_so_far):
        nonlocal parameters, last_history_length
        history = np.asarray(prc_so_far, dtype=float)
        if last_history_length is not None and history.shape[1] != last_history_length + 1:
            reset_state()
        last_history_length = history.shape[1]
        if history.shape[1] < lookback_days:
            return np.zeros(history.shape[0], dtype=int)
        if parameters is None:
            calibration = history[:, -lookback_days:]
            parameters = []
            for idx_a, idx_b, _, _ in indexed_pairs:
                log_a, log_b = np.log(calibration[idx_a]), np.log(calibration[idx_b])
                beta, intercept = np.polyfit(log_b, log_a, 1)
                spread = log_a - (intercept + beta * log_b)
                parameters.append((beta, intercept, spread.mean(), spread.std()))

        current_prices = history[:, -1]
        target = np.zeros(history.shape[0], dtype=int)
        for i, (idx_a, idx_b, _, _) in enumerate(indexed_pairs):
            beta, intercept, spread_mean, spread_std = parameters[i]
            if spread_std < 1e-12 or abs(beta) < 1e-12:
                state[i], held_days[i] = 0, 0
                continue
            spread_now = np.log(current_prices[idx_a]) - (intercept + beta * np.log(current_prices[idx_b]))
            z_score = (spread_now - spread_mean) / spread_std
            next_state = state[i]
            if state[i] == 0:
                next_state = 1 if z_score <= -entry_z else (-1 if z_score >= entry_z else 0)
            elif state[i] == 1 and z_score >= -exit_z:
                next_state = 0
            elif state[i] == -1 and z_score <= exit_z:
                next_state = 0
            if state[i] != 0 and held_days[i] >= max_holding_days:
                next_state = 0
            held_days[i] = 0 if next_state == 0 else (1 if state[i] == 0 else held_days[i] + 1)
            state[i] = next_state
            if next_state:
                scale = min(DOLLAR_LIMIT / current_prices[idx_a], DOLLAR_LIMIT / (abs(beta) * current_prices[idx_b]))
                target[idx_a] += int(np.round(next_state * scale))
                target[idx_b] += int(np.round(-next_state * scale * beta))
        limits = (DOLLAR_LIMIT / current_prices).astype(int)
        target = np.clip(target, -limits, limits).astype(int)
        target[0] = 0
        return target

    get_position.reset_state = reset_state
    return get_position

def evaluate_pair_book(pair_book, lookback_days=ADDITION_LOOKBACK, entry_z=ADDITION_ENTRY_Z, exit_z=ADDITION_EXIT_Z, max_holding_days=ADDITION_MAX_HOLDING_DAYS):
    strategy = make_fixed_beta_strategy(pair_book, lookback_days, entry_z, exit_z, max_holding_days)
    result = run_backtest(prices, strategy, eval_start=DISCOVERY_END, eval_end=VALIDATION_END)
    return compute_metrics(result)

def has_overlap(pair, used_tickers):
    return not set(pair).isdisjoint(used_tickers)

## 2. Validate candidate additions

Every candidate is first scored against the same fixed core. Conflicting candidates are then selected by a global maximum-weight matching on their validation contribution — not by the order in which rows appear.

In [54]:
from functools import lru_cache

core_tickers = {ticker for pair in core_pairs for ticker in pair}
core_metrics = evaluate_pair_book(core_pairs)
addition_rows = []

# Score every candidate relative to the unchanged core before making any choice.
for row in candidate_queue.itertuples():
    pair = (row.ticker_a, row.ticker_b)
    record = {'pair': row.pair, 'ticker_a': row.ticker_a, 'ticker_b': row.ticker_b, 'diagnostic_score': row.diagnostic_score, 'return_correlation': row.return_correlation}
    if has_overlap(pair, core_tickers):
        record.update(decision='ineligible — overlaps initial core', standalone_score=np.nan, expanded_score=np.nan, score_change=np.nan)
    else:
        standalone = evaluate_pair_book([pair])
        expanded = evaluate_pair_book(core_pairs + [pair])
        score_change = expanded['score'] - core_metrics['score']
        record.update(decision='eligible', standalone_score=standalone['score'], expanded_score=expanded['score'], score_change=score_change)
    addition_rows.append(record)

addition_results = pd.DataFrame(addition_rows)
eligible = addition_results.loc[addition_results['decision'].eq('eligible') & addition_results['score_change'].gt(0)].copy().reset_index(drop=True)

# Exact maximum-weight matching: choose the globally strongest set of positive
# validation contributions while allowing each ticker in at most one pair.
candidate_tickers = sorted(set(eligible['ticker_a']) | set(eligible['ticker_b']))
ticker_bits = {ticker: 1 << index for index, ticker in enumerate(candidate_tickers)}
edge_masks = [(ticker_bits[row.ticker_a] | ticker_bits[row.ticker_b]) for row in eligible.itertuples()]
edge_weights = eligible['score_change'].to_numpy()

@lru_cache(maxsize=None)
def best_matching(edge_index, used_mask):
    if edge_index == len(edge_masks):
        return 0.0, ()
    skip_score, skip_edges = best_matching(edge_index + 1, used_mask)
    if edge_masks[edge_index] & used_mask:
        return skip_score, skip_edges
    take_score, take_edges = best_matching(edge_index + 1, used_mask | edge_masks[edge_index])
    take_score += edge_weights[edge_index]
    return (take_score, (edge_index,) + take_edges) if take_score > skip_score else (skip_score, skip_edges)

_, chosen_indices = best_matching(0, 0)
chosen_pairs = [tuple(eligible.iloc[index][['ticker_a', 'ticker_b']]) for index in chosen_indices]
selected_pair_book = list(core_pairs) + chosen_pairs
selected_pair_keys = {f'{a}-{b}' for a, b in chosen_pairs}
addition_results.loc[addition_results['pair'].isin(selected_pair_keys), 'decision'] = 'selected — global validation match'
addition_results.loc[addition_results['decision'].eq('eligible') & ~addition_results['pair'].isin(selected_pair_keys), 'decision'] = 'not selected — weaker conflicting/global combination'

current_metrics = evaluate_pair_book(selected_pair_book)
display(addition_results.style.format({'diagnostic_score': '{:.3f}', 'return_correlation': '{:+.3f}', 'standalone_score': '{:.2f}', 'expanded_score': '{:.2f}', 'score_change': '{:+.2f}'}).set_caption('All candidate results — order-independent validation selection'))

assert len({ticker for pair in selected_pair_book for ticker in pair}) == 2 * len(selected_pair_book)
selected_pair_table = pd.DataFrame(selected_pair_book, columns=['ticker_a', 'ticker_b'])
display(selected_pair_table.style.set_caption('Frozen pair book after global validation matching — no ticker repeats'))
print(f'Validation score of frozen book: {current_metrics["score"]:.2f}')

,pair,ticker_a,ticker_b,diagnostic_score,return_correlation,decision,standalone_score,expanded_score,score_change
0,SMAH-ILVX,SMAH,ILVX,0.920,+0.299,selected — global validation match,28.74,61.47,+31.97
1,NWIG-AENO,NWIG,AENO,0.894,+0.253,selected — global validation match,32.82,65.75,+36.25
2,HUXZ-ACAC,HUXZ,ACAC,0.838,+0.250,selected — global validation match,31.27,64.02,+34.53
3,NGTE-EORC,NGTE,EORC,0.766,+0.231,selected — global validation match,11.70,45.88,+16.38
4,ULXY-HETT,ULXY,HETT,0.739,+0.147,selected — global validation match,30.27,63.26,+33.76
5,EELT-CTGI,EELT,CTGI,0.735,+0.246,selected — global validation match,56.91,90.20,+60.70
6,NWIG-CUBO,NWIG,CUBO,0.708,+0.338,not selected — weaker conflicting/global combination,9.94,43.82,+14.32
7,MTNS-MSDP,MTNS,MSDP,0.662,+0.228,selected — global validation match,13.24,46.45,+16.96
8,AENO-CUBO,AENO,CUBO,0.655,+0.372,not selected — weaker conflicting/global combination,3.88,37.20,+7.71
9,NWIG-DUCT,NWIG,DUCT,0.627,+0.238,not selected — weaker conflicting/global combination,17.89,50.77,+21.27


,ticker_a,ticker_b
0,MHRM,EAFC
1,ACIX,ITPA
2,SMAH,ILVX
3,NWIG,AENO
4,HUXZ,ACAC
5,NGTE,EORC
6,ULXY,HETT
7,EELT,CTGI
8,MTNS,MSDP
9,RTTH,NAYO


Validation score of frozen book: 307.15


## 3. Export the frozen pair book

The exported file contains the validated, non-overlapping book only. Notebook 06 uses this exact file for both fixed-beta and rolling-beta tuning.

In [55]:
validated_pair_book = pd.DataFrame(selected_pair_book, columns=['ticker_a', 'ticker_b'])
validated_pair_book['ticker_a_index'] = validated_pair_book['ticker_a'].map(ticker_to_index)
validated_pair_book['ticker_b_index'] = validated_pair_book['ticker_b'].map(ticker_to_index)
validated_pair_book['validation_start_day'] = DISCOVERY_END + 1
validated_pair_book['validation_end_day'] = VALIDATION_END

validated_path = repo_root / 'research_outputs' / 'validated_pair_book.csv'
validated_pair_book.to_csv(validated_path, index=False)
display(validated_pair_book.style.set_caption('Frozen pair book passed to notebook 06'))
print(f'Exported {len(validated_pair_book)} pairs to {validated_path.relative_to(repo_root)}')

,ticker_a,ticker_b,ticker_a_index,ticker_b_index,validation_start_day,validation_end_day
0,MHRM,EAFC,49,50,501,700
1,ACIX,ITPA,31,43,501,700
2,SMAH,ILVX,10,46,501,700
3,NWIG,AENO,20,1,501,700
4,HUXZ,ACAC,8,27,501,700
5,NGTE,EORC,45,13,501,700
6,ULXY,HETT,40,7,501,700
7,EELT,CTGI,37,25,501,700
8,MTNS,MSDP,33,12,501,700
9,RTTH,NAYO,18,35,501,700


Exported 12 pairs to research_outputs/validated_pair_book.csv


## Validation conclusion

The pair book is now frozen. Tune fixed-beta and rolling-beta configurations in notebook 06, then choose a model before the days 701–1000 evaluation.

In [56]:
print('PAIR_BOOK = [')
for ticker_a, ticker_b in selected_pair_book:
    print(f'    ({ticker_to_index[ticker_a]}, {ticker_to_index[ticker_b]}, {ticker_a!r}, {ticker_b!r}),')
print(']')

PAIR_BOOK = [
    (49, 50, 'MHRM', 'EAFC'),
    (31, 43, 'ACIX', 'ITPA'),
    (10, 46, 'SMAH', 'ILVX'),
    (20, 1, 'NWIG', 'AENO'),
    (8, 27, 'HUXZ', 'ACAC'),
    (45, 13, 'NGTE', 'EORC'),
    (40, 7, 'ULXY', 'HETT'),
    (37, 25, 'EELT', 'CTGI'),
    (33, 12, 'MTNS', 'MSDP'),
    (18, 35, 'RTTH', 'NAYO'),
    (41, 36, 'BLBT', 'FWWG'),
    (42, 11, 'BENI', 'NPCK'),
]


## Next step

- The accepted book contains only pairs that passed discovery and improved the running validation portfolio.
- No stock is used in more than one pair.
- Notebook 06 compares fixed-beta and rolling-beta parameters using only days 501–700.
- Only after choosing one final model should you run the test on days 701–1000.